<a href="https://colab.research.google.com/github/AdamDunn1437/4010_A1/blob/adam-branch/Assignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## AISE4010- Assignment3 - Time Series Classification using TCN and Transformer + Hyperparameter Tuning

## Grade: 100 points

### Instructions

#### Follow These Steps before submitting your assignment

1. Complete the notebook.

2. Make sure all plots have axis labels.

3. Once the notebook is complete, `Restart` your kernel by clicking 'Kernel' > 'Restart & Run All'.

4. Fix any errors until your notebook runs without any problems.

5. Submit one completed notebook for the group to OWL by the deadline.

6. Make sure to reference all external code and documentation used.

### Dataset

The dataset is a sample of 46 satellite images, collected in 2006, located in southwestern France near Toulouse. It
is a 24 km × 24 km area and the dataset uses 3 output classes (2 available) for arable soil classification based on the following paper: https://arxiv.org/pdf/1811.10166.

You will be using helper functions below to prepare it for deep learning models.

In [2]:
# Call this helper method by passing in the names of the provided training and test sets' files.
def read_SITS_data(name_file):
    data = pd.read_table(name_file, sep=',', header=None)

    y_data = data.iloc[:,0]
    y = np.asarray(y_data.values, dtype='uint8')
    y[y>1] = 0

    polygonID_data = data.iloc[:,1]
    polygon_ids = polygonID_data.values
    polygon_ids = np.asarray(polygon_ids, dtype='uint16')

    X_data = data.iloc[:,2:]
    X = X_data.values
    X = np.asarray(X, dtype='float32')

    return  X, polygon_ids, y

In [3]:
def custom_feature_scaling(train, test):
    min_per = np.percentile(train, 2, axis=(0,1))
    max_per = np.percentile(train, 100-2, axis=(0,1))

    new_train = (train-min_per)/(max_per-min_per)
    new_test = (test-min_per)/(max_per-min_per)

    return new_train, new_test

### Question 1 - Data Preprocessing (15%)
- Q1.1 Call "read_SITS_data()" for the training set and store the results as X_train, polygon_ids_train, and y_train.
- Q1.2 Call "read_SITS_data()" above for the test set and store the results as X_test, polygon_ids_test, and y_test.
- Q1.3 Reshape the training and test sets.
  - Each set must be reshaped into a 3-D array. The first dimension will be the number of rows of the original set. The second dimension will be int(x / 3), where x is the number of columns of the original set and int() is a casting function. The third dimension will be 3 (number of channels).
- Q1.4 Call "custom_feature_scaling()" with the training and test sets. Save the results as the final sets for use.
- Q1.5 How many entries are in the training set? How many time steps are in each entry? How many features are there for each time step? How many labels for each entry?


In [9]:
#1.2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#call read SITS data function
X_train, polygon_ids_train, y_train = read_SITS_data('train_dataset.csv')
X_test, polygon_ids_test, y_test = read_SITS_data('test_dataset.csv')

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape: ", y_test.shape)


X_train shape: (260, 447)
X_test shape:  (260, 447)
y_train shape: (260,)
y_test shape:  (260,)


In [10]:
#1.3
#reshape data 3D array [samples, timesteps, channels]
X_train = X_train.reshape(X_train.shape[0], int(X_train.shape[1]/3), 3)
X_test = X_test.reshape(X_test.shape[0], int(X_test.shape[1]/3), 3)

print("X_train reshape:", X_train.shape)
print("X_test reshape: ", X_test.shape)



X_train reshape: (260, 149, 3)
X_test reshape:  (260, 149, 3)


In [11]:
#1.4
#apply custom feature scaling

X_train_new, X_test_new = custom_feature_scaling(X_train, X_test)

print("X_train_new shape:", X_train_new.shape)
print("X_test_new shape:", X_test_new.shape)

X_train_new shape: (260, 149, 3)
X_test_new shape: (260, 149, 3)


*Write your Answer to Q1.5 here:*



### Question2 - Temporal Convolutional Network
- Q2.1 Create a Sequential model for classification. The model should have a TCN layer of size 64, a fully connected layer of size 256, a dropout of 0.3, and a fully connected output layer with Softmax activation (Hint: the logits axis should be on 0). Train the model using the provided dataset for 20 epochs. Use the batch_size of 32, and ADAM optimizer. Print the model summary.
- Q2.2 Train the model with the same parameters, print the model summary and evaluate the model's accuracy on the test set. Print the accuracy.
- Q2.3 Why do we use the Softmax activation on the output layer? In what scenarios does this contrast to using ReLU instead?


In [13]:
!pip install keras-tcn --quiet

In [15]:
from IPython.core import history
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.optimizers import Adam
from tcn import TCN

np.random.seed(17)

#use the scaled data
X_train_tcn = X_train_new
X_test_tcn = X_test_new

#number of classes (2). just printing to double check
num_classes = len(np.unique(y_train))
print("Number of Classes:" , num_classes)

model_tcn = Sequential()
model_tcn.add(
    TCN(
        nb_filters=64,
        input_shape=(X_train_tcn.shape[1], X_train_tcn.shape[2]),
    )
)
model_tcn.add(Dense(256, activation='relu'))
model_tcn.add(Dropout(0.3))
model_tcn.add(Dense(num_classes, activation='softmax'))

#compile model
model_tcn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

#print model summary
model_tcn.summary()

#train
history_tcn = model_tcn.fit(
    X_train_tcn,
    y_train,
    batch_size=32,
    epochs=20,
    verbose=1
)

test_loss, test_acc = model_tcn.evaluate(X_test_tcn, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

Number of Classes: 2


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ tcn_1 (TCN)                     │ (None, 64)             │       136,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 153,922 (601.26 KB)

 Trainable params: 153,922 (601.26 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 7s 121ms/step - accuracy: 0.7673 - loss: 1.9292
Epoch 2/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 120ms/step - accuracy: 0.9079 - loss: 0.4739
Epoch 3/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 118ms/step - accuracy: 0.9012 - loss: 0.2829
Epoch 4/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 204ms/step - accuracy: 0.9303 - loss: 0.2265
Epoch 5/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 147ms/step - accuracy: 0.9343 - loss: 0.1846
Epoch 6/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 121ms/step - accuracy: 0.9385 - loss: 0.1512
Epoch 7/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 121ms/step - accuracy: 0.9477 - loss: 0.1310
Epoch 8/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 120ms/step - accuracy: 0.9372 - loss: 0.1212
Epoch 9/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 125ms/step - accuracy: 0.9881 - loss: 0.0542
Epoch 10/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 125ms/step - accuracy: 0.9603 - loss: 0.1160
Epoch 11/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step - accuracy: 0.9872 - loss: 0.0519
Epoch 12/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 121ms/step - accuracy: 0.9856 - lo

*Write your Answer to Q2.3 Here:*




### Question 3 - Transformer Model
- Q3.1 Create a transformer encoder block. It should use MultiHeadAttention for residual connection. The projection layers can be two Conv1D layers, based on number of feed forward dimensions and with kernel sizes of 1.
- Q3.2 Define the model. It should have 4 encoder blocks, each with 256 heads and feed forward dimensions of 4. Add a flatten layer, then a fully connected layer of size 2 and a fully connected output layer.
- Q3.3 Print the model summary, train the model using 50 epochs and a batch size of 32. Evaluate the model accuracy on the test set and print it.


### Question 4 - Hyperparameter Tuning
- Q4.1 Define a search space for the number of neurons in the fully connected layer that follows the flatten layer. The lower bound should be 2, the upper bound should be 16, and it should search every other value in between. Also have the tuner decide whether or not a dropout layer of 0.3 should be added after the aforementioned layer.
- Q4.2 Using GridSearch, search for the best hyperparameters with respect to accuracy over 50 epochs.
- Q4.3 Using the best hyperparameters, rebuild the model and print the model accuracy.

### Question 6 - Discussion (5%)
- Q6.1 Indicate other hyperparameters relevant to transformers that can be tuned.
- Q6.2 What are the advantages and disadvantages of using GridSearch for finding optimal hyperparameters?


*Write your Answer to Q6.1 and Q6.2 Here:*
